In [1]:
import tiktoken
import torch
from torch.nn.functional import cross_entropy
from torch.utils.data import TensorDataset, DataLoader, random_split

import numpy as np
import pandas as pd
import plotly.express as px
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

if torch.cuda.is_available():
    torch.set_default_device("cuda")

from gpt_arcitecture import download_and_load_gpt2, load_params, generate, GPTModel

settings, vocab, params = download_and_load_gpt2("124M", "./archive")
settings

2025-11-04 21:07:34.406807: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-04 21:07:34.611082: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-04 21:07:35.678858: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-04 21:07:35.679275: I external/local_xla/xla/tsl/cuda/cudart

File already exists and is up-to-date: ./archive/124M/checkpoint
File already exists and is up-to-date: ./archive/124M/encoder.json
File already exists and is up-to-date: ./archive/124M/hparams.json
File already exists and is up-to-date: ./archive/124M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: ./archive/124M/model.ckpt.index
File already exists and is up-to-date: ./archive/124M/model.ckpt.meta
File already exists and is up-to-date: ./archive/124M/vocab.bpe


{'n_vocab': 50257,
 'n_ctx': 1024,
 'n_embd': 768,
 'n_head': 12,
 'n_layer': 12,
 'drop_rate': 0.1,
 'qkv_bias': True}

In [2]:
params["b"].shape, params["g"].shape, params["wpe"].shape, params["wte"].shape
# final norm shift, final norm scale , positionsal encoding weights, vocab vector embeddings/ final linear output head weights

((768,), (768,), (1024, 768), (50257, 768))

In [3]:
len(params["blocks"]), params["blocks"][0].keys() #all transformer blocks, first transfromer layer
# multi-head attention layer, norm 1, norm 2, feed forward network

(12, dict_keys(['attn', 'ln_1', 'ln_2', 'mlp']))

In [4]:
model = GPTModel()
load_params(model, params)

answer = generate(
    model=model,
    sentence= "The different type of AI recommendation algorithms in very concise words are as follows:",
    max_new_tokens=30,
    top_k=20,
    temperature= 1.4,
    context_size=settings["n_ctx"]
)
print(answer)

The different type of AI recommendation algorithms in very concise words are as follows: Randomized trial: When random selection and selection bias (or a particular subset of biases) occur, there is a strong expectation of "safe" results


In [5]:
tokenizer = tiktoken.get_encoding("gpt2")
path = "./archive/News_Category_Dataset_v2.json"
device = torch.get_default_device()
generator = torch.Generator(device = device)

with open(path, "r", encoding="utf-8") as f:
    dicts = []
    json_lines = f.readlines()
    for line in json_lines:
        dicts.append(eval(line))

    data_df = pd.DataFrame(dicts)
    
data_df = data_df[data_df["headline"] != ""]
data_df.reindex(axis=0)
data_df.sample(1, replace=True)

def clean_str(s):
    if not isinstance(s, str):
        return s
    # replace invalid bytes with �
    return s.encode("utf-8", "replace").decode("utf-8")

data_df[["headline"]] = data_df[["headline"]].applymap(clean_str)
data_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 200847 entries, 0 to 200852
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   category           200847 non-null  object
 1   headline           200847 non-null  object
 2   authors            200847 non-null  object
 3   link               200847 non-null  object
 4   short_description  200847 non-null  object
 5   date               200847 non-null  object
dtypes: object(6)
memory usage: 10.7+ MB


In [6]:
data_df = data_df[["category", "headline"]]

fig = px.bar(data_df["category"].value_counts())
fig.show()

data_df["tokens"] = data_df["headline"].map(tokenizer.encode)
data_df["tokens"] = data_df["tokens"].str.slice(0,30)
fig = px.histogram(data_df["tokens"].map(len))
fig.show()

In [7]:
pad_token_id = 50256
for i in range(len(data_df["tokens"])):
    row = data_df.iloc[i]
    pad_times = 30 - len(row["tokens"])
    pad_list = pad_times*[pad_token_id]
    data_df.iloc[i,2].extend(pad_list)

input_df = list(data_df["tokens"].map(list))
output_df = pd.get_dummies(data_df["category"], dtype=int)

labels = list(output_df.columns)

input_tensors = torch.tensor(input_df)
target_tensors = torch.tensor(np.array(output_df))

dataset = TensorDataset(input_tensors, target_tensors)

train_split = 0.9
train_size = int(train_split * len(dataset))
test_size = len(dataset) - train_size
batch_size= 50

train_dataset, test_dataset = random_split(dataset, [train_size, test_size], generator=generator)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True, generator = generator)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, drop_last=True, generator = generator)
len(train_loader), len(test_loader)

(3615, 401)

In [8]:
for param in model.parameters():
    param.requires_grad = False

for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True

for param in model.final_norm.parameters():
    param.requires_grad = True

model.out_head = torch.nn.Linear(settings["n_embd"], len(labels))

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
model_path = "./archive/new_category_classifier.pth"
history_path = "./archive/news_category.csv"
history = []

In [9]:
try:
    # raise Exception()
    model.load_state_dict(torch.load(model_path)["model_state_dict"])
    optimizer.load_state_dict(torch.load(model_path)["optim_state_dict"])
    history_df = pd.read_csv(history_path)
    print("LOADED PRE TRAINED MODEL at", model_path)
    epochs = range(0)
except:
    epochs = range(30)

    for epoch in epochs:
        total_train_loss = 0
        train_correct = 0

        for input_batch, target_batch in tqdm(train_loader):
            
            optimizer.zero_grad()

            output_batch = model(input_batch)[:,-1]
            loss = cross_entropy(output_batch, target_batch.to(torch.float32))
            loss.backward()

            optimizer.step()

            with torch.no_grad():
                total_train_loss += loss.item()
                for pred, trgt in zip(output_batch, target_batch):
                    train_correct += trgt[pred == pred.max()].item()

        avg_train_loss = total_train_loss/(len(train_loader)*batch_size)
        train_accuracy = train_correct/(len(train_loader)*batch_size)
        
        with torch.no_grad():

            history.append({
                "Epoch" : epoch+1,
                "Training Loss": avg_train_loss,
                "Accuracy": train_accuracy
            })

            print(f"Epoch : {epoch+1} : Train Loss = {avg_train_loss:.4f}, Accuracy = {train_accuracy*100:.2f}%\n")

    torch.save({"model_state_dict" : model.state_dict(), "optim_state_dict":optimizer.state_dict()}, model_path)
    history_df = pd.DataFrame(history)
    history_df.to_csv(history_path, index=False)

LOADED PRE TRAINED MODEL at ./archive/new_category_classifier.pth


In [10]:
total_test_loss = 0
test_correct = 0

model.eval()
with torch.no_grad():
    for input_batch, target_batch in tqdm(test_loader):

        output_batch = model(input_batch)[:,-1]
        loss = cross_entropy(output_batch, target_batch.to(torch.float32))
        
        total_test_loss += loss.item()
        for pred, trgt in zip(output_batch, target_batch):
            test_correct += trgt[pred == pred.max()].item()

avg_train_loss = total_test_loss/(len(test_loader)*batch_size)
test_accuracy = test_correct/(len(test_loader)*batch_size)

print(f"Test Loss = {avg_train_loss:.4f}, Test Accuracy = {test_accuracy*100:.2f}%\n")

history_df["Test Loss"] = avg_train_loss
history_df["Test Accuracy"] = test_accuracy

fig = px.line(history_df, "Epoch", ["Training Loss", "Test Loss"])
fig.show()
fig = px.line(history_df, "Epoch", ["Accuracy", "Test Accuracy"])
fig.show()

100%|██████████| 401/401 [00:22<00:00, 17.90it/s]

Test Loss = 0.0548, Test Accuracy = 56.37%



In [18]:
def classify(headline):
    with torch.no_grad():
        model.eval()
        # model.train()
        tokens = tokenizer.encode(headline)[:30]
        tokens += (30 - len(tokens)) * [pad_token_id]

        prediction = model(tokens)[:,-1]/1.4
        probs = torch.softmax(prediction, dim=-1)
        return output_df.columns[prediction.argmax().item()], probs[0].cpu().detach()

def display(probs):
    arr = np.array(probs, dtype=float)
    labs = np.array(output_df.columns)

    # Get top 3
    order = np.argsort(arr)[::-1]
    top_idx = order[:3]
    other_sum = arr[order[3:]].sum()

    # Combine top 3 + 'Other'
    final_labels = labs[top_idx].tolist() + (["Other"] if other_sum > 0 else [])
    final_values = arr[top_idx].tolist() + ([other_sum] if other_sum > 0 else [])

    # Create pie chart
    fig = px.pie(
        names=final_labels,
        values=final_values,
        hole=0.4,  # donut style
        title="Headline's Category Distribution"
    )
    fig.show()

pred, probs = classify("the 2025 LA datacon conference will take place in California State university, Long beach")
display(probs)


pred, probs = classify("Actress Diane Ladd dies at 89: What is known about the Oscar winner")
display(probs)